# Import and Process BRFSS Data

In [4]:
import pandas as pd
import numpy as np
import os
import io
import zipfile
import requests

In [ ]:
import os

# Auto detects project root whether CWD is the project root or notebook/.
if os.path.basename(os.getcwd()) == "notebook":
    _PROJECT_ROOT = os.path.dirname(os.getcwd())
elif os.path.isdir(os.path.join(os.getcwd(), "src")):
    _PROJECT_ROOT = os.getcwd()
else:
    raise FileNotFoundError(
        "Cannot find project root. Run from the project root or notebook/ directory."
    )

DATA_DIR = os.path.join(_PROJECT_ROOT, "data")
os.makedirs(DATA_DIR, exist_ok=True)

# Input:  BRFSS_Dataset_2024.csv (downloaded automatically by the next code cell)
# Output: processed_data_v2.csv will be written to data/.

### Download BRFSS 2024 from CDC
Downloads the raw SAS transport file (~1 GB), extracts selected columns, and saves as CSV to `data/`. Skips automatically if the file already exists.

In [8]:
URL = "https://www.cdc.gov/brfss/annual_data/2024/files/LLCP2024XPT.zip"
ZIP_PATH = os.path.join(DATA_DIR, "LLCP2024.zip")
OUTPUT_PATH = os.path.join(DATA_DIR, "BRFSS_Dataset_2024.csv")

COLS = [
    '_MICHD', '_TOTINDA',
    '_SEX', '_AGEG5YR', '_RACE', 'MARITAL', 'VETERAN3', 'PREGNANT',
    '_EDUCAG', '_INCOMG1', 'SDHBILLS', 'SDHTRNSP', 'HOWSAFE1', 'PRIMINS2',
    '_RFHLTH', 'PHYSHLTH', 'MENTHLTH', 'POORHLTH',
    'CVDSTRK3', '_RFBMI5',
    'DIABETE4', 'DIABTYPE', 'INSULIN1', 'FEETSORE',
    '_SMOKER3', 'USENOW3', '_CURECI3', 'LCSFIRST', 'LCSNUMCG', '_LCSYSMK', '_LCSYQTS', 'STOPSMK2',
    'DRNKANY6', 'AVEDRNK4', '_RFBING6', '_RFDRHV9', '_DRNKWK3', 'MAXDRNKS',
    'CHCCOPD3', 'CHCKDNY2', 'HAVARTH4', '_LTASTH1', 'ADDEPEV3', 'CHCSCNC1', 'CHCOCNC1',
    'LSATISFY', 'SDLONELY',
    'DEAF', 'BLIND', 'DECIDE', 'DIFFWALK', 'DIFFDRES', 'DIFFALON',
    '_HLTHPL2', 'PERSDOC3', 'MEDCOST1', 'CHECKUP1',
    'FLUSHOT7', 'PNEUVAC4',
    'SSBSUGR2',
    'ACEDEPRS', 'ACEDRINK', 'ACEDRUGS', 'ACEPRISN', 'ACEDIVRC', 'ACEPUNCH', 'ACEHURT1', 'ACEADSAF',
    'CIMEMLO1', 'CDSOCIA1',
    '_BMI5', 'HTIN4',
    'MSCODE',
]

if not os.path.exists(OUTPUT_PATH):
    print("Downloading BRFSS 2024 to disk (~1 GB)...")
    with requests.get(URL, stream=True) as r:
        r.raise_for_status()
        with open(ZIP_PATH, 'wb') as f:
            for chunk in r.iter_content(chunk_size=8192):
                f.write(chunk)
    print("Download complete. Converting XPT to CSV...")

    with zipfile.ZipFile(ZIP_PATH) as z:
        xpt_files = [f for f in z.namelist()]
        with z.open(xpt_files[0]) as f:
            first_chunk = True
            for i, chunk in enumerate(pd.read_sas(f, format="xport", chunksize=50_000)):
                available = [c for c in COLS if c in chunk.columns]
                chunk[available].to_csv(OUTPUT_PATH, mode='w' if first_chunk else 'a', header=first_chunk, index=False)
                first_chunk = False
                print(f"  chunk {i+1} written")
                del chunk

    os.remove(ZIP_PATH)
    print(f"Saved to {OUTPUT_PATH}")
else:
    print(f"BRFSS_Dataset_2024.csv already exists at {OUTPUT_PATH}, skipping download.")

BRFSS_Dataset_2024.csv already exists at c:\Users\18327\Desktop\Python Projects\model_myocardial_infarction\data\BRFSS_Dataset_2024.csv, skipping download.


## Curate the list of columns we're pulling in

In [9]:
# TARGET VARIABLE
target = ['_MICHD']  # Ever had CHD or MI

# TREATMENT VARIABLE (for uplift model)
treatment = ['_TOTINDA']  # Leisure-time physical activity in past 30 days

# FEATURES

# Demographics ---
demographic_cols = [
    '_SEX',         # Sex
    '_AGEG5YR',     # Age in 5-year bands
    '_RACE',        # Race/ethnicity
    'MARITAL',      # Marital status — proxy for social support
    'VETERAN3',     # Veteran status
    'PREGNANT',     # Pregnancy status
]

# Socioeconomic / Social Determinants ---
socioeconomic_cols = [
    '_EDUCAG',      # Education level (binned)
    '_INCOMG1',     # Income level (binned)
    'SDHBILLS',     # Unable to pay bills
    'SDHTRNSP',     # Transportation barriers
    'HOWSAFE1',     # Perceived neighborhood safety
    'PRIMINS2',     # Healthcare Coverage
]

# General Health Status ---
health_status_cols = [
    '_RFHLTH',      # Binary good/poor health
    'PHYSHLTH',     # Days physical health not good (past 30 days)
    'MENTHLTH',     # Days mental health not good (past 30 days)
    'POORHLTH',     # Days poor health limited usual activities
]

# Cardiovascular Risk Factors ---
# Note: CVDINFR4 and CVDCRHD4 are dropped since they compose _MICHD (target leakage)
cardiovascular_cols = [
    'CVDSTRK3',     # History of stroke
    '_RFBMI5',      # Overweight/obese flag
]

# Diabetes (potentially strong MI risk factor) ---
diabetes_cols = [
    'DIABETE4',     # Ever told had diabetes
    'DIABTYPE',     # Type 1 or Type 2
    'INSULIN1',     # Currently on insulin — severity proxy
    'FEETSORE',     # Foot sores — diabetes complication proxy
]

# Smoking (potentially strong MI risk factor) ---
smoking_cols = [
    '_SMOKER3',     # 4-level smoking status (daily/someday/former/never)
    'USENOW3',      # Smokeless tobacco use
    '_CURECI3',     # Current e-cigarette user
    'LCSFIRST',     # Age started smoking — exposure duration proxy
    'LCSNUMCG',     # Average cigarettes per day — dose proxy
    '_LCSYSMK',     # Years smoked
    '_LCSYQTS',     # Years since quitting
    'STOPSMK2',     # Attempted to quit in past 12 months
]

# Alcohol ---
alcohol_cols = [
    'DRNKANY6',     # Any drinking in past 30 days
    'AVEDRNK4',     # Average drinks per drinking day
    '_RFBING6',     # Binge drinker flag
    '_RFDRHV9',     # Heavy drinker flag
    '_DRNKWK3',     # Total drinks per week
    'MAXDRNKS',     # Max drinks on a single occasion
]

# Comorbidities ---
comorbidity_cols = [
    'CHCCOPD3',     # COPD/emphysema/chronic bronchitis
    'CHCKDNY2',     # Kidney disease
    'HAVARTH4',     # Arthritis
    '_LTASTH1',     # Ever had asthma
    'ADDEPEV3',     # Depressive disorder
    'CHCSCNC1',     # Non-melanoma skin cancer history
    'CHCOCNC1',     # Melanoma or other cancer history
]

# Mental Health / Wellbeing ---
mental_health_cols = [
    'LSATISFY',     # Life satisfaction
    'SDLONELY',     # Loneliness
]

# Functional Limitations ---
functional_cols = [
    'DEAF',         # Serious hearing difficulty
    'BLIND',        # Serious vision difficulty
    'DECIDE',       # Difficulty concentrating or making decisions
    'DIFFWALK',     # Difficulty walking or climbing stairs
    'DIFFDRES',     # Difficulty dressing or bathing
    'DIFFALON',     # Difficulty doing errands alone
]

# Healthcare Access ---
healthcare_access_cols = [
    '_HLTHPL2',     # Has health insurance
    'PERSDOC3',     # Has a personal doctor
    'MEDCOST1',     # Couldn't afford doctor in past 12 months
    'CHECKUP1',     # Time since last routine checkup
]

# Preventive Care ---
preventive_cols = [
    'FLUSHOT7',     # Flu shot — proxy for general health engagement
    'PNEUVAC4',     # Pneumonia vaccine — proxy for general health engagement
]

# Diet ---
diet_cols = [
    'SSBSUGR2',     # Sugary soda consumption — diet quality proxy
]

# Adverse Childhood Experiences ---
# These may be linked to adult CV outcomes via chronic stress
ace_cols = [
    'ACEDEPRS',     # Lived with someone depressed/mentally ill
    'ACEDRINK',     # Lived with problem drinker
    'ACEDRUGS',     # Lived with drug user
    'ACEPRISN',     # Lived with someone incarcerated
    'ACEDIVRC',     # Parents divorced/separated
    'ACEPUNCH',     # Witnessed domestic violence
    'ACEHURT1',     # Physically hurt by parent/adult
    'ACEADSAF',     # Had protective adult (positive buffer)
]

# Cognitive Decline (potential confounder) ---
cognitive_cols = [
    'CIMEMLO1',     # Worsening memory/thinking difficulties
    'CDSOCIA1',     # Cognitive difficulties affecting work/social life
]

# Body Measures ---
body_cols = [
    '_BMI5',        # Continuous BMI
    'HTIN4',        # Height in inches — may be needed for other computations
]

# Survey Geography (optional — use if modeling regional variation) ---
geography_cols = [
    'MSCODE',       # Metropolitan status (center city/suburban/non-MSA)
]

In [10]:
curated_cols = target + treatment + demographic_cols + socioeconomic_cols + health_status_cols + cardiovascular_cols + diabetes_cols + smoking_cols + alcohol_cols + comorbidity_cols + mental_health_cols + functional_cols + healthcare_access_cols + preventive_cols + diet_cols + ace_cols + cognitive_cols + body_cols + geography_cols

Load Data

In [11]:
import_path = os.path.join(DATA_DIR, "BRFSS_Dataset_2024.csv")
df_raw = pd.read_csv(import_path, usecols=curated_cols)

In [ ]:
# Remove rows with nan for target variable. Only removes ~300 records
df = df_raw[~df_raw['_MICHD'].isna()].copy()
df

,_MICHD,_TOTINDA,_SEX,_AGEG5YR,_RACE,MARITAL,VETERAN3,PREGNANT,_EDUCAG,_INCOMG1,...,ACEPRISN,ACEDIVRC,ACEPUNCH,ACEHURT1,ACEADSAF,CIMEMLO1,CDSOCIA1,_BMI5,HTIN4,MSCODE
0,2.0,1.0,2.0,12.0,1.0,3.0,2.0,NaN,2.0,9.0,...,NaN,NaN,NaN,NaN,NaN,2.0,NaN,2249.0,64.0,1.0
1,1.0,1.0,1.0,13.0,1.0,1.0,1.0,NaN,4.0,7.0,...,NaN,NaN,NaN,NaN,NaN,2.0,NaN,2583.0,70.0,1.0
2,2.0,1.0,1.0,8.0,1.0,6.0,1.0,NaN,3.0,9.0,...,NaN,NaN,NaN,NaN,NaN,2.0,NaN,2253.0,78.0,5.0
3,2.0,1.0,1.0,13.0,1.0,1.0,2.0,NaN,4.0,4.0,...,NaN,NaN,NaN,NaN,NaN,2.0,NaN,2509.0,68.0,5.0
4,2.0,2.0,1.0,6.0,1.0,5.0,2.0,NaN,3.0,2.0,...,NaN,NaN,NaN,NaN,NaN,2.0,NaN,1977.0,68.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
457665,2.0,1.0,1.0,12.0,2.0,1.0,2.0,NaN,1.0,9.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
457666,2.0,2.0,1.0,10.0,2.0,5.0,2.0,NaN,1.0,4.0,...,2.0,2.0,1.0,1.0,1.0,NaN,NaN,2066.0,66.0,NaN
457667,2.0,2.0,1.0,12.0,2.0,5.0,2.0,NaN,4.0,5.0,...,2.0,2.0,1.0,1.0,5.0,NaN,NaN,2437.0,69.0,NaN
457668,2.0,1.0,1.0,6.0,1.0,1.0,2.0,NaN,3.0,6.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2441.0,72.0,NaN


### Survey values are not consistent nor coherent\. Manipulating the values here\.

In [13]:
mapping = {
    1: 1,
    2: 0
}

df['_MICHD'] = df['_MICHD'].map(mapping).fillna(np.nan).astype('float')

In [14]:
mapping = {
    1: 1,
    2: 0,
    9: np.nan
}

df['_TOTINDA'] = df['_TOTINDA'].map(mapping).fillna(np.nan).astype('float')

In [15]:
mapping = {
    1: 1,
    2: 0
}

df['_ismale'] = df['_SEX'].map(mapping).fillna('Unknown').astype('int')
df.drop('_SEX', axis=1, inplace=True)

In [16]:
mapping = {
    1: 'White only, non-Hispanic',
    2: 'Black only, non-Hispanic',
    3: 'AI/AN only, non-Hispanic',
    4: 'Asian only, non-Hispanic',
    5: 'NH/PI only, non-Hispanic',
    6: 'Other race only, non-Hispanic',
    7: 'Multiracial, non-Hispanic',
    8: 'Hispanic',
    9: "Unknown/Refused"
}

df['_RACE'] = df['_RACE'].map(mapping).fillna('Unknown/Refused').astype('category')

In [17]:
mapping = {
    1: 'Married',
    2: 'Divorced',
    3: 'Widowed',
    4: 'Separated',
    5: 'Never married',
    6: 'Unmarried couple',
    9: 'Refused'
}

df['MARITAL'] = df['MARITAL'].map(mapping).fillna('Unknown').astype('category')


In [18]:
mapping = {1: 'Yes', 2: 'No', 7: 'Unknown', 9: 'Refused'}

for col in [
    'VETERAN3', 'PREGNANT', 'SDHBILLS', 'SDHTRNSP', 
    'CVDSTRK3', 'INSULIN1', 'FEETSORE', 'STOPSMK2',
    'DRNKANY6', '_RFBING6', '_RFDRHV9', 'CHCCOPD3',
    'CHCKDNY2', 'HAVARTH4', 'ADDEPEV3', 'CHCSCNC1',
    'CHCOCNC1', 'DEAF', 'BLIND', 'DECIDE', 'DIFFWALK',
    'DIFFDRES', 'DIFFALON', 'MEDCOST1', 'FLUSHOT7',
    'PNEUVAC4', 'ACEDEPRS', 'ACEDRINK', 'ACEDRUGS',
    'ACEPRISN', 'CIMEMLO1', 'CDSOCIA1'
]:
    df[col] = df[col].map(mapping).fillna('Unknown').astype('category')

In [19]:
mapping = {1: 'No High', 2: 'Grad High', 3: 'Attend College', 4: 'Grad College', 9: 'Unknown'}
df['_EDUCAG'] = df['_EDUCAG'].map(mapping).fillna('Unknown').astype('category')

In [20]:
mapping = {
    1: 'Less than $15,000',
    2: '$15,000 to < $25,000',
    3: '$25,000 to < $35,000',
    4: '$35,000 to < $50,000',
    5: '$50,000 to < $100,000',
    6: '$100,000 to < $200,000',
    7: '$200,000 or more',
    9: 'Unknown'
}

df['_INCOMG1'] = df['_INCOMG1'].map(mapping).fillna('Unknown').astype('category')

In [21]:
mapping = {
    1: 'Extremely Safe',
    2: 'Safe',
    3: 'Unsafe',
    4: 'Extremely Unsafe',
    7: 'Unknown',
    9: 'Refused'
}

df['HOWSAFE1'] = df['HOWSAFE1'].map(mapping).fillna('Unknown').astype('category')

In [22]:
mapping = {
    1: 'Employer or union plan',
    2: 'Private plan purchased on own',
    3: 'Medicare',
    4: 'Medigap',
    5: 'Medicaid',
    6: 'CHIP',
    7: 'Military (TRICARE/VA/CHAMP-VA)',
    8: 'Indian Health Service',
    9: 'State-sponsored health plan',
    10: 'Other government program',
    88: 'No coverage of any type',
    77: "Unknown",
    99: 'Refused'
}

df['PRIMINS2'] = df['PRIMINS2'].map(mapping).fillna('Unknown').astype('category')

In [23]:
mapping = {
    1: 'Good or Better Health',
    2: 'Fair or Poor Health',
    9: 'Unknown/Refused'
}

df['_RFHLTH'] = df['_RFHLTH'].map(mapping).fillna('Unknown/Refused').astype('category')

In [24]:
cols = ['PHYSHLTH', 'MENTHLTH', 'POORHLTH', 'MAXDRNKS']
df[cols] = df[cols].replace({77: np.nan, 88: np.nan, 99: np.nan}).astype('float')

In [25]:
cols = ['_DRNKWK3']
df[cols] = df[cols].replace({99900: np.nan}).astype('float')

In [26]:
mapping = {
    1: 'Under 25',
    2: 'Over 25',
    9: 'Unknown/Refused'
}

df['_RFBMI5'] = df['_RFBMI5'].map(mapping).fillna('Unknown/Refused').astype('category')

In [27]:
mapping = {
    1: 'Diabetes',
    2: 'Diabetes, Pregnancy',
    3: 'No Diabetes',
    4: 'Borderline Diabetes',
    7: 'Unknown',
    9: 'Refused'
}

df['DIABETE4'] = df['DIABETE4'].map(mapping).fillna('Unknown').astype('category')

In [28]:
mapping = {1: 'Type 1', 2: 'Type 2', 7: 'Unknown', 9: 'Refused'}

df['DIABTYPE'] = df['DIABTYPE'].map(mapping).fillna('Unknown').astype('category')

In [29]:
mapping = {1: 'Everyday Smoker', 2: 'Someday Smoker', 3:'Former Smoker', 4:'Non Smoker', 9: 'Unknown/Refused'}

df['_SMOKER3'] = df['_SMOKER3'].map(mapping).fillna('Unknown/Refused').astype('category')

In [30]:
mapping = {1: 'Everyday', 2: 'Someday', 3:'Not at all', 7:'Unknown', 9: 'Refused'}

df['USENOW3'] = df['USENOW3'].map(mapping).fillna('Unknown').astype('category')

In [31]:
mapping = {1: 'Not User', 2: 'Current User', 9: 'Unknown/Refused'}

df['_CURECI3'] = df['_CURECI3'].map(mapping).fillna('Unknown/Refused').astype('category')

In [32]:
# It's okay to convert never smokers to nan here because that information is already captured in _SMOKER3
cols = ['LCSFIRST', 'LCSNUMCG']
df[cols] = df[cols].replace({777: np.nan, 888: np.nan, 999: np.nan}).astype('float')

In [33]:
cols = ['AVEDRNK4']
df[cols] = df[cols].replace({77: np.nan, 88: 0, 99: np.nan}).astype('float')

In [34]:
mapping = {1: 'No', 2: 'Yes', 9: 'Unknown/Refused'}

for col in [
    '_LTASTH1'
]:
    df[col] = df[col].map(mapping).fillna('Unknown/Refused').astype('category')

In [35]:
mapping = {
    1: 'Very Satisfied',
    2: 'Satisfied',
    3: 'Dissatisfied',
    4: 'Very Dissatisfied',
    7: 'Unknown',
    9: 'Refused'
}

df['LSATISFY'] = df['LSATISFY'].map(mapping).fillna('Unknown').astype('category')

In [36]:
mapping = {
    1: 'Always',
    2: 'Usually',
    3: 'Sometimes',
    4: 'Rarely',
    5: 'Never',
    7: 'Unknown',
    9: 'Refused'
}

df['SDLONELY'] = df['SDLONELY'].map(mapping).fillna('Unknown').astype('category')

In [37]:
mapping = {
    1: 'Have Insurance',
    2: 'No Insurance',
    9: 'Unknown/Refused'
}

df['_HLTHPL2'] = df['_HLTHPL2'].map(mapping).fillna('Unknown/Refused').astype('category')

In [38]:
mapping = {
    1: 'One',
    2: 'More than one',
    3: 'No',
    7: 'Unknown',
    9: 'Refused'
}

df['PERSDOC3'] = df['PERSDOC3'].map(mapping).fillna('Unknown').astype('category')

In [39]:
mapping = {
    1: 'Within 1',
    2: 'Within 2',
    3: 'Within 5',
    4: '5 or more',
    7: 'Unknown',
    8: 'Never',
    9: 'Refused'
}

df['CHECKUP1'] = df['CHECKUP1'].map(mapping).fillna('Unknown').astype('category')

In [40]:
cols = ['SSBSUGR2']
df[cols] = df[cols].replace({777: np.nan, 888: 0, 999: np.nan}).astype('float')

In [41]:
mapping = {1: 'Yes', 2: 'No', 7: 'Unknown', 8: 'Parents Not Married', 9: 'Refused'}

df['ACEDIVRC'] = df['ACEDIVRC'].map(mapping).fillna('Unknown').astype('category')

In [42]:
mapping = {1: 'Never', 2: 'Once', 3: 'More than Once', 7: 'Unknown', 9: 'Refused'}

for col in [
    'ACEPUNCH', 'ACEHURT1'
]:
    df[col] = df[col].map(mapping).fillna('Unknown').astype('category')

In [43]:
mapping = {1: 'Never', 2: 'Little', 3: 'Some', 4: 'Most', 5: 'All', 7: 'Unknown', 9: 'Refused'}

df['ACEADSAF'] = df['ACEADSAF'].map(mapping).fillna('Unknown').astype('category')

In [44]:
mapping_mscode = {1:'Center City MSA', 3:'Suburban MSA', 5:'Not MSA'}

df['MSCODE'] = df['MSCODE'].map(mapping_mscode).astype('category')

In [45]:
def convert_freq_to_daily(series):
    s = series.copy()
    result = pd.Series(np.nan, index=s.index, dtype=float)

    daily = s.between(101, 199)
    weekly = s.between(201, 299)
    monthly = s.between(301, 399)
    never = s == 888

    result[daily] = s[daily] - 100
    result[weekly] = (s[weekly] - 200) / 7
    result[monthly] = (s[monthly] - 300) / 30
    result[never] = 0
    # 777 (don't know) and 999 (refused) fall through to NaN

    return result

# Standardize Timeframe to days
df['SSBSUGR2'] = convert_freq_to_daily(df['SSBSUGR2'])

### Fix SAS XPT near-zero sentinel values
The SAS transport format encodes certain missing/zero values as extremely small floats (e.g. `5.4e-79`) rather than true zeros. This affects columns like `_DRNKWK3` and `_LCSYSMK`. The cell below detects and replaces these artifacts with `0`.

In [46]:
for col in df.select_dtypes("number").columns:
    bad = (df[col] != 0) & (df[col].abs() < 1e-10)
    if bad.any():
        print(f"  {col}: {bad.sum():,} near-zero sentinels → 0")
        df.loc[bad, col] = 0

print("Sentinel cleanup complete.")

  _LCSYSMK: 1,527 near-zero sentinels → 0
  _LCSYQTS: 8,103 near-zero sentinels → 0
  _DRNKWK3: 200,784 near-zero sentinels → 0
  MAXDRNKS: 1 near-zero sentinels → 0
Sentinel cleanup complete.


In [48]:
output_path = os.path.join(DATA_DIR, "processed_data.csv")
df.to_csv(output_path, index=False)
print(f"Saved to {output_path}")

Saved to c:\Users\18327\Desktop\Python Projects\model_myocardial_infarction\data\processed_data.csv


<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=88479c56-07bc-4e04-92b4-d8345dc226f4' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>